In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/diabetic_data.csv")
df_processed = df.copy()

# weight isn't here because we are going to drop it.

columns_with_missing_vals = [
    "race",
    "payer_code",
    "medical_specialty",
    "race",
    "diag_3",
    "diag_2",
    "diag_1"
]

for column in columns_with_missing_vals:
    df_processed[column] = df_processed[column].replace("?", "Unknown")

In [ ]:
# Checking if the values in these columns changed to "unknown", which is what we want.
# They should return 0 questionmarks for every on of them.
(df_processed == "?").sum()

In [ ]:
##############################################################################################
# Remove columns with little predictive value, such as ids and 'weight' with almost no data. 
# The `weight` feature contains approximately 97% missing values.
# Since processing such a large proportion would introduce significant uncertainty, the column 
# is removed from the dataset.
##############################################################################################

df_processed = df_processed.drop(columns=["weight", "patient_nbr", "encounter_id"])

# Originally: (101766, 50)
print(df_processed.shape)


In [ ]:
## Encodings:

# TARGET MUST BE CONVERTED TO BINARY
# <30 days -> 1
# >30 days -> 0
# No days -> 0

"""
gender -> One-Hot Encoding
race -> One-Hot Encoding
diabetesMed -> Label Encoding (Y/N -> 1/0)
change -> Label Encoding (Change/No)
insulin -> One hot Encoding 
age -> Ordinal Encoding (because age groups have a natural order)
"""

# Create the binary target for readmitted less than 30 days
# 1-> Readmitted within 30 days
# 0 -> Other/Not readmitted within 30 days
df_processed["readmitted_binary"] = (
    df_processed["readmitted"] == "<30"
).astype(int)

df_processed["readmitted_binary"].value_counts()





In [ ]:
##############################################################################################
# Remove the old target column, now we are using the readmitted binary column encoding
##############################################################################################

df_processed.drop(columns=["readmitted"], inplace=True)

In [ ]:
# Age values have a natural order

# We can use ordinal Mapping

age_mapping = {
    "[0-10]" : 0,
    "[10-20]" : 1,
    "[20-30]" : 2,
    "[30-40]" : 3,
    "[40-50]" : 4,
    "[50-60]" : 5,
    "[60-70]" : 6,
    "[70-80]" : 7,
    "[80-90]" : 8,
    "[90-100]" : 9
}

df_processed["age"] = df_processed["age"].map(age_mapping)

In [ ]:
## Binary Yes/No Columns
# Same idea as above

df_processed["change"] = (
    df_processed["change"]
    .map({
        "No" : 0,
        "Ch" : 1
    })
)

In [ ]:
df_processed["diabetesMed"] = (
    df_processed["diabetesMed"]
    .map({
        "No" : 0,
        "Yes" : 1
    })
)

In [ ]:
df_processed["gender"].value_counts()

df_processed["gender"] = (
    df_processed["gender"].map({
        "Female" : 0,
        "Male" : 1
    })
)

In [ ]:
# one-hot encoding of this data.
# they contain more than two classes.
df_processed = pd.get_dummies(
    df_processed,
    columns=[
        "race",
        "insulin",
        "payer_code",
        "medical_specialty"
    ],
    drop_first=True
)

In [ ]:
# X(Features) -> 2D matrix, each row a data sample, each col a feature
# y(Target) -> 1D vector of correct answers corresponding to each row in X.

X = df_processed.drop(columns=["readmitted_binary"]) 
y = df_processed["readmitted_binary"] 

from sklearn.model_selection import train_test_split

# Split between training and testing phase: 80/20 rule
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)


In [ ]:

"""
We want to understand what each categorical variable represents so that we can apply the correct encoding to each.

"""
categorical_cols = df_processed.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"\n{col}")
    print(df_processed[col].unique())

In [ ]:
# print(X_train.info())
print(X_train.isnull().sum().sum())
print(y_train.value_counts(normalize=True))